# PSE EDGE Filing Scraper

This notebook scrapes corporate filings from the Philippine Stock Exchange EDGE portal and delivers them as organized PDF files in a ZIP archive.

**Features:**
- Search by company name or ticker symbol
- Filter by date range and filing types
- Automatic checkpoint/resume for long downloads
- Organized folder structure with standardized filenames

---

## Step 1: Setup and Installation
Run this cell first to install required packages.

In [ ]:
# Install required packages
!pip install -q requests beautifulsoup4 lxml weasyprint

print("Setup complete!")

## Step 2: Import Libraries and Configuration

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import json
import os
import io
import zipfile
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, asdict
from urllib.parse import urljoin, quote
import time
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files

# Configuration
BASE_URL = "https://edge.pse.com.ph"
CHECKPOINT_DIR = "/content/checkpoints"
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, "checkpoint.json")
REQUEST_DELAY = 0.5  # seconds between requests to avoid rate limiting
MAX_RETRIES = 3

# Ensure checkpoint directory exists
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Libraries loaded successfully!")

## Step 3: Filing Type Definitions

In [ ]:
# Master list of 83 filing types
# Structure: form_no -> (short_name, full_name, is_common, quarterly_type)
# quarterly_type: None, 'backward', or 'forward'

FILING_TYPES = {
    # Common Filings
    "17-1": ("AnnualRpt", "Annual Report", True, None),
    "17-2": ("QuarterlyRpt", "Quarterly Report", True, "backward"),
    "4-29": ("DisburseRpt", "Disbursement of Proceeds and Progress Report", True, "backward"),
    "4-30": ("MatInfo", "Material Information/Transactions", True, None),
    "4-31": ("PressRel", "Press Release", True, None),
    "6-1": ("CashDiv", "Declaration of Cash Dividends", True, None),
    "6-2": ("StockDiv", "Declaration of Stock Dividends", True, None),
    
    # Corporate Actions
    "4-1": ("AcqDisp-Assets", "Acquisition or Disposition of Assets", True, None),
    "4-2": ("AcqDisp-Shares", "Acquisition or Disposition of Shares", True, None),
    "4-3": ("Amend-AOI", "Amendments to Articles of Incorporation", True, None),
    "4-4": ("Amend-BL", "Amendments to By-Laws", True, None),
    "4-5": ("ChgControl", "Change in Control of Issuer", False, None),
    "4-8": ("ChgDirOff", "Change in Directors and/or Officers", True, None),
    "4-23": ("MergerCons", "Mergers and Consolidations", False, None),
    "5-1": ("SubAcq", "Substantial Acquisitions", False, None),
    
    # Meetings & Governance
    "4-24": ("ASM-Results", "Results of Annual/Special Stockholders' Meeting", True, None),
    "4-25": ("BOD-Results", "Results of Organizational Meeting of BOD", True, None),
    "7-1": ("ASM-Notice", "Notice of Annual/Special Stockholders' Meeting", True, None),
    "7-2": ("ASM-Postpone", "Postponement of Annual Stockholders' Meeting", False, None),
    "I-ACGR": ("CorpGov", "Integrated Annual Corporate Governance Report", True, None),
    
    # Ownership & Securities
    "17-6": ("InitBenOwn", "Initial Statement of Beneficial Ownership", False, None),
    "17-7": ("ChgBenOwn", "Statement of Changes in Beneficial Ownership", True, None),
    "17-8": ("Own5Pct", "Report by Owner of More Than Five Percent", False, None),
    "17-11": ("StockholdersLst", "List of Stockholders", False, None),
    "17-12": ("Top100Stockholders", "List of Top 100 Stockholders", True, "forward"),
    "17-13": ("ForeignOwn", "Foreign Ownership Report", True, None),
    "POR-1": ("PubOwn", "Public Ownership Report", True, "forward"),
    "POR-2": ("PubOwnClass", "Public Ownership Report (Classified Shares)", False, "forward"),
    
    # Securities Issuance
    "4-14": ("RightsOffer", "Stock Rights Offering", False, None),
    "4-15": ("NewEquity", "Creation and Issuance of New Equity Security", False, None),
    "4-16": ("DebtIssue", "Issuance of Debt Securities", False, None),
    "4-17": ("Warrants", "Issuance of Warrants", False, None),
    "4-18": ("Options", "Options", False, None),
    
    # Share Transactions
    "9-1": ("BuyBack", "Share Buy-Back Transactions", True, None),
    "9-2": ("TreasurySale", "Sale of Treasury Shares", False, None),
    "4-11": ("ChgShares", "Change in Number of Issued/Outstanding Shares", False, None),
    "4-19": ("Declassify", "Declassification of Shares", False, None),
    "4-20": ("Reclassify", "Reclassification of Shares", False, None),
    "4-21": ("Redemption", "Redemption of Security", False, None),
    
    # Change Notifications
    "4-6": ("ChgContact", "Change in Corporate Contact Details/Website", False, None),
    "4-7": ("ChgName", "Change in Corporate Name/Stock Symbol", False, None),
    "4-9": ("ChgAuditor", "Change in External Auditor", False, None),
    "4-10": ("ChgFiscalYr", "Change in Fiscal Year", False, None),
    "4-12": ("ChgParVal", "Change in Par Value", False, None),
    "12-1": ("ChgSTA", "Change in Stock Transfer Agent", False, None),
    "13-1": ("ChgDirHoldings", "Change in Shareholdings of Directors/Officers", False, None),
    
    # Communications
    "4-13": ("ClarNews", "Clarification of News Reports", True, None),
    "4-32": ("ExchReply", "Reply to Exchange's Query", False, None),
    "14-1": ("InvBriefing", "Notice of Analysts'/Investors' Briefing", False, None),
    "16-1": ("CorpUpdate", "Update on Corporate Actions/Transactions", False, None),
    
    # Trading
    "4-33": ("VolHalt", "Voluntary Trading Halt", False, None),
    "4-34": ("VolSuspend", "Voluntary Trading Suspension", False, None),
    "CMIC-2": ("UPM-Reply", "Reply to Inquiry on Unusual Price Movement", False, None),
    
    # Legal & Regulatory
    "4-26": ("LegalProc", "Legal Proceedings", False, None),
    "4-27": ("OfferComp", "Notification of Completion/Termination of Offering", False, None),
    "4-28": ("AuditFindings", "Findings of External Auditor (Fraud and Error)", False, None),
    "17-16": ("TenderOffer", "Tender Offer Report", False, None),
    
    # Other Categories
    "6-3": ("PropDiv", "Declaration of Property Dividends", False, None),
    "10-1": ("SubAcqDisp", "Acquisition/Disposition by Subsidiaries/Affiliates", False, None),
    "11-1": ("VolLockUp", "Voluntary Lock-Up", False, None),
    "BL-1": ("BackdoorList", "Comprehensive Disclosure on Backdoor Listing", False, None),
    "LR-1": ("ShareIssuance", "Comprehensive Disclosure on Issuance of Shares", False, None),
    "LR-2": ("PlaceSub", "Comprehensive Disclosure on Placing/Subscription", False, None),
    "LR-3": ("ListingApp", "Submission of Documents for Listing Applications", False, None),
    "DLR-1": ("VolDelist", "Voluntary Delisting", False, None),
    "DLR-2": ("DelistPetition", "Petition for Voluntary Delisting", False, None),
    "QR-1": ("QuasiReorg", "Quasi-Reorganization", False, None),
    "4-22": ("JointVenture", "Joint Ventures", False, None),
    
    # Reporting Extensions & Other
    "17-3": ("ExtReq-17A", "Request for Extension to File SEC Form 17-A", False, None),
    "17-4": ("ExtReq-17Q", "Request for Extension to File SEC Form 17-Q", False, None),
    "17-5": ("InfoStatement", "Information Statement", False, None),
    "17-9": ("InstOwn5Pct", "Short Form Report by Institutional Owners", False, None),
    "17-10": ("NumShareholders", "Report on the Number of Shareholders", False, None),
    "17-14": ("MGB-Verify", "Annual Verification of Mines and Geosciences Bureau", False, None),
    "17-15": ("DOE-Verify", "Annual Verification of Department of Energy", False, None),
    "17-18": ("OtherSEC", "Other SEC Forms, Reports and Requirements", False, None),
    "CP-TR1": ("CPTechRpt", "CP Technical Report", False, None),
    
    # ETF Forms (Rare)
    "ETF-1": ("ETF-DailyTrade", "Daily Trading Information", False, None),
    "ETF-2": ("ETF-MonthlyRpt", "Monthly Issuance and Redemption Report", False, None),
    "ETF-3": ("ETF-Dividend", "Dividend Distribution", False, None),
    "ETF-4": ("ETF-CreateShares", "Creation of Shares", False, None),
    "ETF-5": ("ETF-RedeemShares", "Redemption of Shares", False, None),
    "ETF-6": ("ETF-ChgAuthPart", "Change in Authorized Participant", False, None),
    "ETF-7": ("ETF-ChgCustodian", "Change in Custodian", False, None),
    "ETF-8": ("ETF-ChgFundMgr", "Change in Fund Manager", False, None),
    "ETF-9": ("ETF-ChgIdxProv", "Change in Index Provider", False, None),
    "ETF-10": ("ETF-ChgMktMkr", "Change in Market Maker", False, None),
    "ETF-11": ("ETF-ChgTA", "Change in Transfer Agent", False, None),
    "ETF-12": ("ETF-MatInfo", "Material Information", False, None),
}

# Create reverse lookup from short name to form number
SHORT_NAME_TO_FORM = {v[0]: k for k, v in FILING_TYPES.items()}

# Get common and rare filing types
COMMON_FILINGS = {k: v for k, v in FILING_TYPES.items() if v[2]}
RARE_FILINGS = {k: v for k, v in FILING_TYPES.items() if not v[2]}

print(f"Loaded {len(FILING_TYPES)} filing types ({len(COMMON_FILINGS)} common, {len(RARE_FILINGS)} rare)")

In [ ]:
# PSE Listed Companies Master List
# Format: (company_id, ticker, company_name)
# This list is hardcoded to avoid API dependency issues

PSE_COMPANIES = [
    ("29", "2GO", "2GO Group, Inc."),
    ("626", "HOUSE", "8990 Holdings, Inc."),
    ("13", "ABG", "A Brown Company, Inc."),
    ("14", "ANS", "A. Soriano Corporation"),
    ("114", "ABS", "ABS-CBN Corporation"),
    ("15", "ABSP", "ABS-CBN Holdings Corporation"),
    ("643", "AGF", "AG Finance, Incorporated"),
    ("177", "APC", "APC Group, Inc."),
    ("56", "ATN", "ATN Holdings, Inc."),
    ("174", "ABA", "AbaCore Capital Holdings, Inc."),
    ("16", "AEV", "Aboitiz Equity Ventures, Inc."),
    ("609", "AP", "Aboitiz Power Corporation"),
    ("33", "AR", "Abra Mining and Industrial Corporation"),
    ("48", "ACE", "Acesite (Phils.) Hotel Corporation"),
    ("619", "ANI", "AgriNurture, Inc."),
    ("212", "AGI", "Alliance Global Group, Inc."),
    ("602", "FOOD", "Alliance Select Foods International, Inc."),
    ("121", "ALC", "Alsons Consolidated Resources, Inc."),
    ("620", "ALTR", "Alterra Capital Partners, Inc."),
    ("612", "ALHI", "Anchor Land Holdings, Inc."),
    ("52", "APO", "Anglo Philippine Holdings Corporation"),
    ("178", "APX", "Apex Mining Co., Inc."),
    ("38", "ARA", "Araneta Properties, Inc."),
    ("172", "ALCO", "Arthaland Corporation"),
    ("55", "AAA", "Asia Amalgamated Holdings Corporation"),
    ("641", "AUB", "Asia United Bank Corporation"),
    ("176", "ABB", "Asiabest Group International Inc."),
    ("53", "ATI", "Asian Terminals, Inc."),
    ("34", "AT", "Atlas Consolidated Mining and Development Corporation"),
    ("19", "AB", "Atok-Big Wedge Co., Inc."),
    ("57", "AC", "Ayala Corporation"),
    ("180", "ALI", "Ayala Land, Inc."),
    ("31", "BLFI", "BDO Leasing and Finance, Inc."),
    ("260", "BDO", "BDO Unibank, Inc."),
    ("62", "BHI", "BHI Holdings, Inc."),
    ("234", "BPI", "Bank of the Philippine Islands"),
    ("60", "BSC", "Basic Energy Corporation"),
    ("21", "BEL", "Belle Corporation"),
    ("108", "BC", "Benguet Corporation"),
    ("9", "BRN", "Berjaya Philippines Inc."),
    ("49", "BLOOM", "Bloomberry Resorts Corporation"),
    ("181", "BMM", "Bogo Medellin Milling Company, Inc."),
    ("63", "BH", "Boulevard Holdings, Inc."),
    ("66", "BKR", "Bright Kindle Resources & Investments Inc."),
    ("601", "COL", "COL Financial Group, Inc."),
    ("636", "CAL", "Calata Corporation"),
    ("624", "CEB", "Cebu Air, Inc."),
    ("110", "CHI", "Cebu Holdings, Incorporated"),
    ("182", "CPV", "Cebu Property Ventures and Development Corporation"),
    ("183", "CAT", "Central Azucarera de Tarlac, Inc."),
    ("223", "CEU", "Centro Escolar University"),
    ("652", "CNPF", "Century Pacific Food, Inc."),
    ("621", "CPM", "Century Peak Metals Holdings Corporation"),
    ("189", "CPG", "Century Properties Group, Inc."),
    ("22", "CIP", "Chemical Industries of the Philippines, Inc."),
    ("184", "CHIB", "China Banking Corporation"),
    ("630", "TECH", "Cirtek Holdings Philippines Corporation"),
    ("209", "LAND", "City & Land Developers, Incorporated"),
    ("39", "CLI", "Cityland Development Corporation"),
    ("228", "CSB", "Citystate Savings Bank, Inc."),
    ("637", "COAL", "Coal Asia Holdings Incorporated"),
    ("648", "CIC", "Concepcion Industrial Corporation"),
    ("213", "CHP", "Concrete Aggregates Corporation"),
    ("50", "COSCO", "Cosco Capital, Inc."),
    ("657", "CROWN", "Crown Asia Chemicals Corporation"),
    ("186", "CEI", "Crown Equities, Inc."),
    ("67", "CYBER", "Cyber Bay Corporation"),
    ("639", "DNL", "D&L Industries, Inc."),
    ("187", "DFNN", "DFNN, Inc."),
    ("188", "DMC", "DMCI Holdings, Inc."),
    ("2", "DAVIN", "Da Vinci Capital Holdings, Inc."),
    ("642", "DELM", "Del Monte Pacific Limited"),
    ("647", "DWC", "Discovery World Corporation"),
    ("68", "DIZ", "Dizon Copper-Silver Mines, Inc."),
    ("651", "DD", "DoubleDragon Properties Corp."),
    ("71", "EEI", "EEI Corporation"),
    ("634", "EW", "East West Banking Corporation"),
    ("70", "ECP", "EasyCall Communications Philippines, Inc."),
    ("632", "EMP", "Emperador Inc."),
    ("190", "ELI", "Empire East Land Holdings, Inc."),
    ("603", "EDC", "Energy Development Corporation"),
    ("219", "EURO", "Euro-Med Laboratories Phil., Inc."),
    ("191", "EVER", "Ever-Gotesco Resources and Holdings, Inc."),
    ("171", "EIB", "Export and Industry Bank, Inc."),
    ("225", "FJP", "F & J Prince Holdings Corporation"),
    ("25", "FEU", "Far Eastern University, Incorporated"),
    ("75", "FDC", "Filinvest Development Corporation"),
    ("226", "FLI", "Filinvest Land, Inc."),
    ("196", "FILRT", "Filipino Fund, Inc."),
    ("80", "FYN", "Filsyn Corporation"),
    ("81", "FAF", "First Abacus Financial Holdings Corporation"),
    ("600", "FGEN", "First Gen Corporation"),
    ("649", "FMETF", "First Metro Philippine Equity Exchange Traded Fund, Inc."),
    ("197", "FPH", "First Philippine Holdings Corporation"),
    ("220", "FOR", "Forum Pacific, Inc."),
    ("198", "GEO", "GEOGRACE Resources Philippines, Inc."),
    ("611", "GMA7", "GMA Holdings, Inc."),
    ("610", "GMA", "GMA Network, Inc."),
    ("633", "GTCAP", "GT Capital Holdings, Inc."),
    ("94", "GSMI", "Ginebra San Miguel, Inc."),
    ("224", "FNI", "Global Ferronickel Holdings, Inc."),
    ("193", "GERI", "Global-Estate Resorts, Inc."),
    ("129", "PORT", "Globalport 900, Inc."),
    ("69", "GLO", "Globe Telecom, Inc."),
    ("661", "HVN", "Golden Haven Memorial Park, Inc."),
    ("221", "GPH", "Grand Plaza Hotel Corporation"),
    ("132", "GREEN", "Greenergy Holdings Incorporated"),
    ("644", "TUGS", "Harbor Star Shipping Services, Inc."),
    ("211", "HLCM", "Holcim Philippines, Inc."),
    ("82", "HI", "House of Investments, Inc."),
    ("613", "I", "I-Remit, Inc."),
    ("623", "IPO", "IP E-Game Ventures, Inc."),
    ("4", "IPM", "IPM Holdings, Inc."),
    ("84", "IRC", "IRC Properties, Inc."),
    ("36", "ISM", "ISM Communications Corporation"),
    ("201", "IMP", "Imperial Resources, Inc."),
    ("622", "IMI", "Integrated Micro-Electronics, Inc."),
    ("83", "ICT", "International Container Terminal Services, Inc."),
    ("203", "ION", "Ionics, Inc."),
    ("204", "IS", "Island Information & Technology, Inc."),
    ("660", "IRC", "Italpinas Development Corporation"),
    ("210", "JGS", "JG Summit Holdings, Inc."),
    ("134", "JACK", "Jackstones, Inc."),
    ("86", "JFC", "Jollibee Foods Corporation"),
    ("261", "JOH", "Jolliville Holdings Corporation"),
    ("87", "KEP", "Keppel Philippines Holdings, Inc."),
    ("88", "KPPI", "Keppel Philippines Properties, Inc."),
    ("236", "LBC", "LBC Express Holdings, Inc."),
    ("205", "LMG", "LMG Chemicals Corporation"),
    ("12", "LTG", "LT Group, Inc."),
    ("96", "LR", "Leisure & Resorts World Corporation"),
    ("98", "LC", "Lepanto Consolidated Mining Company"),
    ("227", "LFM", "Liberty Flour Mills, Inc."),
    ("27", "LIB", "Liberty Telecoms Holdings, Inc."),
    ("37", "LODE", "Lodestar Investment Holdings Corporation"),
    ("61", "LPZ", "Lopez Holdings Corporation"),
    ("115", "LSC", "Lorenzo Shipping Corporation"),
    ("126", "MED", "MEDCO Holdings, Inc."),
    ("24", "MJC", "MJC Investments Corporation"),
    ("131", "MRC", "MRC Allied, Inc."),
    ("206", "MHC", "Mabuhay Holdings Corporation"),
    ("100", "MVC", "Mabuhay Vinyl Corporation"),
    ("145", "MACAY", "Macay Holdings, Inc."),
    ("106", "MAC", "MacroAsia Corporation"),
    ("263", "MFIN", "Makati Finance Corporation"),
    ("117", "MBC", "Manila Broadcasting Company"),
    ("1", "MB", "Manila Bulletin Publishing Corporation"),
    ("118", "MER", "Manila Electric Company"),
    ("102", "MJC", "Manila Jockey Club, Inc."),
    ("119", "MA", "Manila Mining Corporation"),
    ("270", "MWC", "Manila Water Company, Inc."),
    ("120", "MFC", "Manulife Financial Corporation"),
    ("175", "MARC", "Marcventures Holdings, Inc."),
    ("135", "MAXS", "Max's Group, Inc."),
    ("627", "MWIDE", "Megawide Construction Corporation"),
    ("127", "MEG", "Megaworld Corporation"),
    ("202", "MCP", "Melco Crown (Philippines) Resorts Corporation"),
    ("3", "MAH", "Metro Alliance Holdings & Equities Corp."),
    ("192", "MGH", "Metro Global Holdings Corporation"),
    ("604", "MPI", "Metro Pacific Investments Corporation"),
    ("659", "MRSGI", "Metro Retail Stores Group, Inc."),
    ("128", "MBT", "Metropolitan Bank & Trust Company"),
    ("105", "MG", "Millennium Global Holdings, Inc."),
    ("606", "NRCP", "National Reinsurance Corporation of the Philippines"),
    ("179", "NXGN", "NextGenesis Corporation"),
    ("103", "NI", "NiHAO Mineral Resources International, Inc."),
    ("625", "NIKL", "Nickel Asia Corporation"),
    ("264", "NOW", "Now Corporation"),
    ("207", "OM", "Omico Corporation"),
    ("616", "ORE", "Oriental Peninsula Resources Group, Inc."),
    ("43", "OPM", "Oriental Petroleum and Minerals Corporation"),
    ("20", "PAL", "PAL Holdings, Inc."),
    ("99", "PCP", "PICOP Resources, Inc."),
    ("8", "PTFC", "PTFC Redevelopment Corporation"),
    ("605", "LOTO", "Pacific Online Systems Corporation"),
    ("109", "PA", "Pacifica, Inc."),
    ("104", "PMC", "Panasonic Manufacturing Philippines Corporation"),
    ("194", "PAX", "Paxys, Inc."),
    ("617", "PIP", "Pepsi-Cola Products Philippines, Inc."),
    ("578", "PERC", "PetroEnergy Resources Corporation"),
    ("136", "PCOR", "Petron Corporation"),
    ("122", "WEB", "PhilWeb Corporation"),
    ("97", "PHC", "Philcomsat Holdings Corporation"),
    ("137", "PX", "Philex Mining Corporation"),
    ("628", "PXP", "Philex Petroleum Corporation"),
    ("208", "PBC", "Philippine Bank of Communications"),
    ("640", "PBB", "Philippine Business Bank"),
    ("138", "PHES", "Philippine Estates Corporation"),
    ("631", "H2O", "Philippine H2O Ventures Corp."),
    ("6", "TEL", "Philippine Long Distance Telephone Company"),
    ("139", "PNB", "Philippine National Bank"),
    ("7", "PNC", "Philippine National Construction Corporation"),
    ("141", "PRC", "Philippine Racing Club, Inc."),
    ("40", "RLT", "Philippine Realty and Holdings Corporation"),
    ("142", "PSB", "Philippine Savings Bank"),
    ("143", "SEVN", "Philippine Seven Corporation"),
    ("76", "PTT", "Philippine Telegraph and Telephone Corporation"),
    ("144", "PTC", "Philippine Trust Company"),
    ("107", "PHN", "Phinma Corporation"),
    ("608", "PNX", "Phoenix Petroleum Philippines, Inc."),
    ("655", "PSPC", "Phoenix Semiconductor Philippines Corp."),
    ("148", "PHA", "Premiere Horizon Alliance Corporation"),
    ("158", "PLC", "Premium Leisure Corp."),
    ("30", "PRIM", "Prime Media Holdings, Inc."),
    ("26", "POPI", "Prime Orion Philippines, Inc."),
    ("111", "PRIM", "Primetown Property Group, Inc."),
    ("214", "PRMX", "Primex Corporation"),
    ("150", "PPC", "Pryce Corporation"),
    ("629", "PGOLD", "Puregold Price Club, Inc."),
    ("77", "RFM", "RFM Corporation"),
    ("153", "REG", "Republic Glass Holdings Corporation"),
    ("232", "RCB", "Rizal Commercial Banking Corporation"),
    ("195", "RLC", "Robinsons Land Corporation"),
    ("646", "RRHI", "Robinsons Retail Holdings, Inc."),
    ("635", "ROCK", "Rockwell Land Corporation"),
    ("64", "ROX", "Roxas Holdings, Inc."),
    ("54", "RCI", "Roxas and Company, Inc."),
    ("658", "SBS", "SBS Philippines Corporation"),
    ("599", "SM", "SM Investments Corporation"),
    ("112", "SMPH", "SM Prime Holdings, Inc."),
    ("161", "SOC", "SOCResources, Inc."),
    ("237", "SPC", "SPC Power Corporation"),
    ("654", "SSI", "SSI Group, Inc."),
    ("222", "STI", "STI Education Systems Holdings, Inc."),
    ("154", "SMC", "San Miguel Corporation"),
    ("151", "FB", "San Miguel Pure Foods Company, Inc."),
    ("156", "SFI", "Seafront Resources Corporation"),
    ("32", "SECB", "Security Bank Corporation"),
    ("157", "SCC", "Semirara Mining and Power Corporation"),
    ("218", "SHNG", "Shang Properties, Inc."),
    ("160", "SGI", "Solid Group, Inc."),
    ("614", "SPH", "Splash Corporation"),
    ("41", "SLI", "Sta. Lucia Land, Inc."),
    ("147", "STR", "Starmalls, Inc."),
    ("164", "STC", "Steniel Manufacturing Corporation"),
    ("78", "SLF", "Sun Life Financial Inc."),
    ("73", "SUN", "Suntrust Home Developers, Inc."),
    ("479", "SCC", "Supercity Realty Development Corporation"),
    ("165", "SFD", "Swift Foods, Inc."),
    ("166", "SGP", "Synergy Grid & Development Phils., Inc."),
    ("163", "TKC", "TKC Metals Corporation"),
    ("478", "PSE", "The Philippine Stock Exchange, Inc."),
    ("45", "OV", "The Philodrill Corporation"),
    ("650", "TFHI", "Top Frontier Investment Holdings, Inc."),
    ("233", "TA", "Trans-Asia Oil and Energy Development Corporation"),
    ("653", "TAPC", "Trans-Asia Petroleum Corporation"),
    ("269", "TBGI", "Transpacific Broadband Group Int'l. Inc."),
    ("645", "RWM", "Travellers International Hotel Group, Inc."),
    ("92", "UNI", "Unioil Resources & Holdings Company, Inc."),
    ("93", "URC", "Universal Robina Corporation"),
    ("170", "VLL", "Vista Land & Lifescapes, Inc."),
    ("656", "VITA", "Vitarich Corporation"),
    ("168", "VUL", "Vulcan Industrial & Mining Corporation"),
    ("169", "WIN", "Waterfront Philippines, Inc."),
    ("167", "WLCON", "Wilcon Depot, Inc."),
    ("662", "WLCON", "Wilcon Depot, Inc."),
    ("663", "MONDE", "Monde Nissin Corporation"),
    ("664", "CNVRG", "Converge ICT Solutions Inc."),
    ("665", "MREIT", "MREIT, Inc."),
    ("666", "AREIT", "AREIT, Inc."),
    ("667", "ACEN", "ACEN Corporation"),
    ("668", "ALLHC", "AllHome Corp."),
    ("669", "FRUIT", "Fruitas Holdings, Inc."),
    ("670", "HOME", "AllHome Corp."),
    ("671", "DITO", "DITO CME Holdings Corp."),
    ("672", "ACEX", "Axelum Resources Corp."),
    ("673", "OGC", "OrtiGas & Company Limited Partnership"),
    ("674", "DDMPR", "DDMP REIT, Inc."),
    ("675", "FILRT", "Filinvest REIT Corp."),
    ("676", "RCR", "RL Commercial REIT, Inc."),
    ("677", "VREIT", "VistaREIT, Inc."),
    ("678", "CREIT", "Citicore Energy REIT Corp."),
    ("679", "PREIT", "Premiere Island Power REIT Corp."),
    ("680", "KEEPR", "The Keepers Holdings, Inc."),
    ("681", "MM", "MerryMart Consumer Corp."),
    ("682", "MEDIC", "Medilines Distributors Incorporated"),
]

# Create lookup dictionaries
COMPANY_BY_ID = {c[0]: {"ticker": c[1], "name": c[2]} for c in PSE_COMPANIES}
COMPANY_BY_TICKER = {c[1]: {"id": c[0], "name": c[2]} for c in PSE_COMPANIES}

print(f"Loaded {len(PSE_COMPANIES)} PSE-listed companies")

In [ ]:
@dataclass
class Company:
    """Represents a PSE-listed company."""
    company_id: str
    company_name: str
    ticker: str

@dataclass
class Disclosure:
    """Represents a single disclosure filing."""
    disclosure_id: str
    company_id: str
    ticker: str
    company_name: str
    template_name: str
    form_number: str
    disclosed_date: str  # YYYY-MM-DD
    subject: str

@dataclass
class Attachment:
    """Represents a PDF attachment."""
    attachment_id: str
    filename: str
    is_main: bool
    amendment_level: int  # 0 = original, 1+ = amendment number

class PSEEdgeClient:
    """Client for interacting with PSE EDGE portal."""
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Connection': 'keep-alive',
            'Referer': 'https://edge.pse.com.ph/',
            'Content-Type': 'application/x-www-form-urlencoded',
        })
        # Use static company list - no API calls needed
        self._companies_cache = [
            Company(company_id=c[0], ticker=c[1], company_name=c[2])
            for c in PSE_COMPANIES
        ]
    
    def _request(self, method: str, url: str, **kwargs) -> requests.Response:
        """Make HTTP request with retry logic."""
        for attempt in range(MAX_RETRIES):
            try:
                time.sleep(REQUEST_DELAY)
                response = self.session.request(method, url, timeout=30, **kwargs)
                response.raise_for_status()
                return response
            except requests.RequestException as e:
                if attempt == MAX_RETRIES - 1:
                    raise
                time.sleep(2 ** attempt)
        return None
    
    def get_companies(self) -> List[Company]:
        """Return the static list of PSE-listed companies."""
        return self._companies_cache
    
    def search_companies(self, query: str) -> List[Company]:
        """Search companies by name or ticker."""
        query = query.upper().strip()
        
        if not query:
            return self._companies_cache[:20]
        
        matches = []
        for c in self._companies_cache:
            if query in c.ticker.upper() or query in c.company_name.upper():
                matches.append(c)
        
        return matches[:50]
    
    def get_disclosures(
        self,
        company_id: str,
        start_date: str,
        end_date: str,
        form_numbers: List[str],
        page: int = 0
    ) -> Tuple[List[Disclosure], int]:
        """Fetch disclosures for a company within date range.
        
        The PSE EDGE API uses form-encoded POST body data.
        """
        url = f"{BASE_URL}/companyDisclosures/search.ax"
        
        # Form data in POST body (as per pse-edge library)
        form_data = {
            'keyword': company_id,
            'tmplNm': '',
        }
        
        try:
            # Use POST with form data in body
            response = self._request('POST', url, data=form_data)
            html = response.text
            soup = BeautifulSoup(html, 'lxml')
            
            disclosures = []
            
            # Get company info from cache
            company_name = ''
            ticker = ''
            for comp in self._companies_cache:
                if comp.company_id == company_id:
                    company_name = comp.company_name
                    ticker = comp.ticker
                    break
            
            # Find disclosure table with class "list"
            table = soup.find('table', {'class': 'list'})
            
            if table:
                # Find tbody and get rows
                tbody = table.find('tbody')
                rows = tbody.find_all('tr') if tbody else table.find_all('tr')
                
                for row in rows:
                    cells = row.find_all('td')
                    if len(cells) >= 3:
                        # Find the disclosure link
                        link = row.find('a', href=True)
                        if link and 'openDiscViewer' in link.get('href', ''):
                            href = link.get('href', '')
                            
                            # Extract edge_no from href
                            edge_match = re.search(r"edge_no=([^'\"&]+)", href)
                            disclosure_id = edge_match.group(1).strip() if edge_match else ''
                            
                            if not disclosure_id:
                                continue
                            
                            # Get disclosure name/subject (first column with link)
                            subject = link.get_text(strip=True)
                            
                            # Get template/form name (second column)
                            template_name = cells[1].get_text(strip=True) if len(cells) > 1 else ''
                            form_number = self._extract_form_number(template_name)
                            
                            # Filter by form numbers if specified
                            if form_numbers and form_number and form_number not in form_numbers:
                                continue
                            
                            # Get date (third column typically)
                            date_text = cells[2].get_text(strip=True) if len(cells) > 2 else ''
                            disclosed_date = self._parse_date(date_text)
                            
                            # Filter by date range
                            if disclosed_date and start_date and end_date:
                                if disclosed_date < start_date or disclosed_date > end_date:
                                    continue
                            
                            disclosures.append(Disclosure(
                                disclosure_id=disclosure_id,
                                company_id=company_id,
                                ticker=ticker,
                                company_name=company_name,
                                template_name=template_name,
                                form_number=form_number,
                                disclosed_date=disclosed_date or date_text,
                                subject=subject
                            ))
            
            # Try to find total pages from pagination
            total_pages = 1
            pagination = soup.find('div', class_='paging')
            if pagination:
                page_links = pagination.find_all('a')
                for link in page_links:
                    try:
                        page_num = int(link.get_text(strip=True))
                        total_pages = max(total_pages, page_num)
                    except ValueError:
                        pass
            
            return disclosures, total_pages
            
        except Exception as e:
            print(f"Error fetching disclosures: {e}")
            import traceback
            traceback.print_exc()
            return [], 0
    
    def _parse_date(self, date_text: str) -> str:
        """Parse date string to YYYY-MM-DD format."""
        if not date_text:
            return ''
        
        # Clean up the date text
        date_text = date_text.strip()
        
        # Try common formats
        formats = [
            '%Y-%m-%d',
            '%m/%d/%Y',
            '%d/%m/%Y', 
            '%B %d, %Y',
            '%b %d, %Y',
            '%Y/%m/%d',
            '%m-%d-%Y',
            '%d-%m-%Y',
        ]
        
        for fmt in formats:
            try:
                dt = datetime.strptime(date_text, fmt)
                return dt.strftime('%Y-%m-%d')
            except ValueError:
                continue
        
        # Try to extract date pattern
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', date_text)
        if match:
            return match.group(0)
        
        # Try MM/DD/YYYY pattern
        match = re.search(r'(\d{1,2})/(\d{1,2})/(\d{4})', date_text)
        if match:
            month, day, year = match.groups()
            return f"{year}-{month.zfill(2)}-{day.zfill(2)}"
        
        return ''
    
    def _extract_form_number(self, template_name: str) -> str:
        """Extract form number from template name."""
        patterns = [
            r'SEC Form ([\d]+-[\d]+)',
            r'Form ([\d]+-[\d]+)',
            r'\b(17-\d+)\b',
            r'\b(\d+-\d+)\b',
            r'(POR-\d+)',
            r'(ETF-\d+)',
            r'(I-ACGR)',
            r'(CMIC-\d+)',
            r'(BL-\d+)',
            r'(LR-\d+)',
            r'(DLR-\d+)',
            r'(QR-\d+)',
            r'(CP-TR\d+)',
        ]
        
        for pattern in patterns:
            match = re.search(pattern, template_name, re.IGNORECASE)
            if match:
                return match.group(1).upper()
        
        return ''
    
    def get_disclosure_attachments(self, disclosure_id: str) -> Tuple[List[Attachment], str]:
        """Fetch attachments for a disclosure."""
        url = f"{BASE_URL}/openDiscViewer.do"
        params = {'edge_no': disclosure_id}
        
        try:
            response = self._request('GET', url, params=params)
            html = response.text
            soup = BeautifulSoup(html, 'lxml')
            
            attachments = []
            
            for link in soup.find_all('a', href=True):
                href = link.get('href', '')
                text = link.get_text(strip=True)
                
                if 'downloadFile.do' in href or ('download' in href.lower() and 'file' in href.lower()):
                    file_id_match = re.search(r'file_id=(\d+)', href)
                    if not file_id_match:
                        file_id_match = re.search(r'id=(\d+)', href)
                    
                    file_id = file_id_match.group(1) if file_id_match else ''
                    
                    if file_id:
                        filename = text or 'document.pdf'
                        if not filename.lower().endswith('.pdf'):
                            filename += '.pdf'
                        
                        amendment_level = 0
                        amend_match = re.search(r'\[Amend-?(\d+)\]', text, re.IGNORECASE)
                        if amend_match:
                            amendment_level = int(amend_match.group(1))
                        elif 'amend' in text.lower():
                            amendment_level = 1
                        
                        parent_section = link.find_parent(['div', 'section', 'table'])
                        is_main = True
                        if parent_section:
                            section_id = parent_section.get('id', '').lower()
                            section_class = ' '.join(parent_section.get('class', [])).lower()
                            if 'attach' in section_id or 'attach' in section_class:
                                is_main = False
                        
                        attachments.append(Attachment(
                            attachment_id=file_id,
                            filename=filename,
                            is_main=is_main,
                            amendment_level=amendment_level
                        ))
            
            seen_ids = set()
            unique_attachments = []
            for att in attachments:
                if att.attachment_id not in seen_ids:
                    seen_ids.add(att.attachment_id)
                    unique_attachments.append(att)
            
            return unique_attachments, html
            
        except Exception as e:
            print(f"Error fetching disclosure {disclosure_id}: {e}")
            return [], ''
    
    def download_attachment(self, attachment_id: str) -> Optional[bytes]:
        """Download a PDF attachment."""
        url = f"{BASE_URL}/downloadFile.do"
        params = {'file_id': attachment_id}
        
        try:
            response = self._request('GET', url, params=params)
            content_type = response.headers.get('content-type', '').lower()
            
            if 'pdf' in content_type or 'octet-stream' in content_type or len(response.content) > 1000:
                return response.content
            return None
        except Exception as e:
            print(f"Error downloading attachment {attachment_id}: {e}")
            return None

# Initialize client
client = PSEEdgeClient()
print(f"PSE EDGE client initialized with {len(client.get_companies())} companies!")

In [ ]:
# DEBUG CELL - Run this to test the API directly
# This will show exactly what the API returns and help diagnose issues

def debug_api_call():
    """Test the disclosure API and show what's returned."""
    import requests
    from bs4 import BeautifulSoup
    
    # Test with Ayala Corporation (company_id=57)
    company_id = "57"
    company_name = "Ayala Corporation"
    
    print(f"Testing API for: {company_name} (ID: {company_id})")
    print("=" * 60)
    
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Referer': 'https://edge.pse.com.ph/',
        'Content-Type': 'application/x-www-form-urlencoded',
    })
    
    url = "https://edge.pse.com.ph/companyDisclosures/search.ax"
    
    # POST with form data in body
    print("\nFetching disclosures...")
    try:
        form_data = {
            'keyword': company_id,
            'tmplNm': '',
        }
        response = session.post(url, data=form_data, timeout=30)
        print(f"Status: {response.status_code}")
        print(f"Content length: {len(response.text)} chars")
        
        soup = BeautifulSoup(response.text, 'lxml')
        
        # Find table
        table = soup.find('table', {'class': 'list'})
        if table:
            print("\nFound table with class 'list'!")
            
            # Get all rows
            rows = table.find_all('tr')
            print(f"Found {len(rows)} rows")
            
            # Show first few rows' HTML structure
            print("\n--- First 3 rows structure ---")
            for i, row in enumerate(rows[:3]):
                print(f"\nRow {i+1}:")
                # Show all links in this row
                links = row.find_all('a')
                for link in links:
                    href = link.get('href', 'NO HREF')
                    onclick = link.get('onclick', 'NO ONCLICK')
                    text = link.get_text(strip=True)[:40]
                    print(f"  Link: '{text}'")
                    print(f"    href: {href}")
                    print(f"    onclick: {onclick[:100] if onclick != 'NO ONCLICK' else onclick}")
            
            # Look for onclick handlers with openDiscViewer
            print("\n--- Looking for disclosure patterns ---")
            
            # Pattern 1: onclick with openDiscViewer
            onclick_links = table.find_all('a', onclick=lambda x: x and 'openDiscViewer' in x)
            print(f"Links with onclick containing 'openDiscViewer': {len(onclick_links)}")
            
            # Pattern 2: onclick with edge_no
            edge_links = table.find_all('a', onclick=lambda x: x and 'edge_no' in str(x))
            print(f"Links with onclick containing 'edge_no': {len(edge_links)}")
            
            # Pattern 3: Any onclick
            any_onclick = table.find_all('a', onclick=True)
            print(f"Links with any onclick: {len(any_onclick)}")
            
            if onclick_links:
                print("\n--- Sample disclosures found! ---")
                for i, link in enumerate(onclick_links[:5]):
                    onclick = link.get('onclick', '')
                    text = link.get_text(strip=True)[:50]
                    print(f"{i+1}. {text}")
                    print(f"   onclick: {onclick}")
                    
                    # Extract edge_no from onclick
                    import re
                    edge_match = re.search(r"edge_no=([^'\"&\s]+)", onclick)
                    if edge_match:
                        print(f"   edge_no: {edge_match.group(1)}")
            elif any_onclick:
                print("\n--- Sample onclick handlers ---")
                for i, link in enumerate(any_onclick[:5]):
                    onclick = link.get('onclick', '')
                    text = link.get_text(strip=True)[:50]
                    print(f"{i+1}. {text}")
                    print(f"   onclick: {onclick}")
        else:
            print("No table with class 'list' found")
            print(f"\nFull response:\n{response.text[:2000]}")
            
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

# Run the debug
print("Running API debug...")
print()
debug_api_call()

## Step 5: File Naming Utilities

In [ ]:
def sanitize_filename(filename: str) -> str:
    """Sanitize filename by removing special characters."""
    # Replace spaces with underscores
    filename = filename.replace(' ', '_')
    # Remove special characters
    filename = re.sub(r'[/\\:*?"<>|]', '', filename)
    # Remove multiple underscores
    filename = re.sub(r'_+', '_', filename)
    # Strip leading/trailing underscores
    filename = filename.strip('_')
    return filename

def extract_quarter_from_filename(filename: str) -> Optional[str]:
    """Try to extract quarter from PDF filename."""
    patterns = [
        r'Q([1-4])\s*20\d{2}',  # Q1 2024
        r'\(Q([1-4])\s*20\d{2}\)',  # (Q3 2024)
        r'17-Q([1-4])',  # 17-Q3
        r'[_\s]Q([1-4])[_\s\.]',  # _Q1_ or .Q1.
        r'^Q([1-4])[_\s]',  # Q1_ at start
    ]
    
    for pattern in patterns:
        match = re.search(pattern, filename, re.IGNORECASE)
        if match:
            return f"Q{match.group(1)}"
    
    return None

def derive_quarter_backward(date_str: str) -> str:
    """Derive quarter using backward method (for 17-2, 4-29).
    
    - Filed Jan-Mar = Q4 of prior year
    - Filed Apr-Jun = Q1
    - Filed Jul-Sep = Q2
    - Filed Oct-Dec = Q3
    """
    date = datetime.strptime(date_str, '%Y-%m-%d')
    month = date.month
    
    if month <= 3:
        return 'Q4'
    elif month <= 6:
        return 'Q1'
    elif month <= 9:
        return 'Q2'
    else:
        return 'Q3'

def derive_quarter_forward(date_str: str) -> str:
    """Derive quarter using forward/calendar method (for POR-1, POR-2, 17-12).
    
    - Filed Jan-Mar = Q1
    - Filed Apr-Jun = Q2
    - Filed Jul-Sep = Q3
    - Filed Oct-Dec = Q4
    """
    date = datetime.strptime(date_str, '%Y-%m-%d')
    month = date.month
    
    if month <= 3:
        return 'Q1'
    elif month <= 6:
        return 'Q2'
    elif month <= 9:
        return 'Q3'
    else:
        return 'Q4'

def get_quarter_for_filing(
    form_number: str,
    disclosure_date: str,
    attachment_filename: str = ''
) -> Optional[str]:
    """Determine quarter for a filing."""
    if form_number not in FILING_TYPES:
        return None
    
    quarterly_type = FILING_TYPES[form_number][3]
    if quarterly_type is None:
        return None
    
    # Try to extract from filename first
    if attachment_filename:
        quarter = extract_quarter_from_filename(attachment_filename)
        if quarter:
            return quarter
    
    # Fall back to date-based derivation
    if quarterly_type == 'backward':
        return derive_quarter_backward(disclosure_date)
    else:
        return derive_quarter_forward(disclosure_date)

def build_filename(
    ticker: str,
    disclosure_date: str,
    form_number: str,
    original_filename: str = '',
    is_single_attachment: bool = True
) -> str:
    """Build standardized filename.
    
    Format: Ticker_Year_Quarter_FilingType_OriginalFilename.pdf
    """
    year = disclosure_date[:4]
    
    # Get short name for filing type
    if form_number in FILING_TYPES:
        short_name = FILING_TYPES[form_number][0]
    else:
        short_name = form_number.replace('-', '')
    
    # Get quarter if applicable
    quarter = get_quarter_for_filing(form_number, disclosure_date, original_filename)
    
    # Build base name
    parts = [ticker, year]
    if quarter:
        parts.append(quarter)
    parts.append(short_name)
    
    # Add original filename if multiple attachments
    if not is_single_attachment and original_filename:
        # Clean up original filename
        clean_name = original_filename
        # Remove extension
        clean_name = re.sub(r'\.pdf$', '', clean_name, flags=re.IGNORECASE)
        # Remove redundant ticker/year/quarter
        clean_name = re.sub(rf'^{ticker}[_\s]*', '', clean_name, flags=re.IGNORECASE)
        clean_name = re.sub(rf'{year}[_\s]*', '', clean_name)
        if quarter:
            clean_name = re.sub(rf'{quarter}[_\s]*', '', clean_name, flags=re.IGNORECASE)
        clean_name = sanitize_filename(clean_name)
        if clean_name:
            parts.append(clean_name)
    
    # For zero-attachment filings, add date
    if not original_filename:
        date_compact = disclosure_date.replace('-', '')
        parts.append(date_compact)
    
    filename = '_'.join(parts) + '.pdf'
    return sanitize_filename(filename)

def build_folder_path(ticker: str, disclosure_date: str, form_number: str) -> str:
    """Build folder path: Ticker/Year/FilingType/"""
    year = disclosure_date[:4]
    
    if form_number in FILING_TYPES:
        short_name = FILING_TYPES[form_number][0]
    else:
        short_name = form_number.replace('-', '')
    
    return os.path.join(ticker, year, short_name)

print("File naming utilities loaded!")

## Step 6: Checkpoint System

In [ ]:
@dataclass
class Checkpoint:
    """Checkpoint for resuming interrupted downloads."""
    # Batch parameters
    company_ids: List[str]
    start_date: str
    end_date: str
    form_numbers: List[str]
    
    # Progress tracking
    downloaded_files: List[str]  # List of completed file paths
    processed_disclosures: List[str]  # List of processed disclosure IDs
    current_company_index: int
    current_page: int
    
    # Metadata
    created_at: str
    updated_at: str

def save_checkpoint(checkpoint: Checkpoint):
    """Save checkpoint to local storage."""
    checkpoint.updated_at = datetime.now().isoformat()
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(asdict(checkpoint), f, indent=2)

def load_checkpoint() -> Optional[Checkpoint]:
    """Load checkpoint from local storage if exists."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r') as f:
                data = json.load(f)
                return Checkpoint(**data)
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
    return None

def clear_checkpoint():
    """Clear existing checkpoint."""
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

def create_checkpoint(
    company_ids: List[str],
    start_date: str,
    end_date: str,
    form_numbers: List[str]
) -> Checkpoint:
    """Create a new checkpoint."""
    now = datetime.now().isoformat()
    return Checkpoint(
        company_ids=company_ids,
        start_date=start_date,
        end_date=end_date,
        form_numbers=form_numbers,
        downloaded_files=[],
        processed_disclosures=[],
        current_company_index=0,
        current_page=0,
        created_at=now,
        updated_at=now
    )

print("Checkpoint system loaded!")

## Step 7: Page Renderer (for zero-attachment filings)

In [ ]:
def render_page_as_pdf(html_content: str, disclosure: Disclosure) -> Optional[bytes]:
    """Render disclosure page as PDF for zero-attachment filings."""
    try:
        from weasyprint import HTML, CSS
        
        # Add some basic styling
        styled_html = f"""
        <!DOCTYPE html>
        <html>
        <head>
            <meta charset="UTF-8">
            <style>
                body {{
                    font-family: Arial, sans-serif;
                    margin: 40px;
                    font-size: 12px;
                }}
                .header {{
                    text-align: center;
                    margin-bottom: 30px;
                    border-bottom: 2px solid #333;
                    padding-bottom: 20px;
                }}
                .header h1 {{
                    color: #003366;
                    margin: 0;
                }}
                .metadata {{
                    margin: 20px 0;
                }}
                .metadata p {{
                    margin: 5px 0;
                }}
                .label {{
                    font-weight: bold;
                    color: #666;
                }}
                .content {{
                    margin-top: 30px;
                }}
                .footer {{
                    margin-top: 40px;
                    font-size: 10px;
                    color: #999;
                    text-align: center;
                }}
            </style>
        </head>
        <body>
            <div class="header">
                <h1>PSE EDGE Disclosure</h1>
            </div>
            <div class="metadata">
                <p><span class="label">Company:</span> {disclosure.company_name} ({disclosure.ticker})</p>
                <p><span class="label">Template:</span> {disclosure.template_name}</p>
                <p><span class="label">Date:</span> {disclosure.disclosed_date}</p>
                <p><span class="label">Subject:</span> {disclosure.subject}</p>
                <p><span class="label">Attachments:</span> (0)</p>
            </div>
            <div class="content">
                <p>This disclosure has no PDF attachments. The original page content is shown below:</p>
                <hr>
            </div>
            <div class="footer">
                <p>Generated from PSE EDGE - https://edge.pse.com.ph</p>
            </div>
        </body>
        </html>
        """
        
        pdf_bytes = HTML(string=styled_html).write_pdf()
        return pdf_bytes
    except Exception as e:
        print(f"Error rendering page as PDF: {e}")
        return None

print("Page renderer loaded!")

## Step 8: Main Scraper Logic

In [ ]:
@dataclass
class DownloadResult:
    """Result of a file download."""
    filepath: str
    content: bytes
    ticker: str
    filing_type: str
    date: str
    is_rendered: bool = False

@dataclass
class ErrorRecord:
    """Record of a download error."""
    ticker: str
    filing_type: str
    date: str
    error_type: str
    message: str

class PSEEdgeScraper:
    """Main scraper class."""
    
    def __init__(self, log_callback=None):
        self.client = PSEEdgeClient()
        self.log_callback = log_callback or print
        self.downloaded_files: List[DownloadResult] = []
        self.errors: List[ErrorRecord] = []
        self.checkpoint: Optional[Checkpoint] = None
        self._filename_counts: Dict[str, int] = {}
    
    def log(self, status: str, ticker: str, filing_type: str, date: str, filename: str):
        """Log progress."""
        timestamp = datetime.now().strftime('%H:%M:%S')
        message = f"[{timestamp}] [{status}] {ticker} - {filing_type} - {date} - {filename}"
        self.log_callback(message)
    
    def _get_unique_filename(self, filepath: str) -> str:
        """Ensure filename is unique by adding suffix if needed."""
        if filepath not in self._filename_counts:
            self._filename_counts[filepath] = 0
            return filepath
        
        self._filename_counts[filepath] += 1
        count = self._filename_counts[filepath]
        
        # Add counter suffix before extension
        base, ext = os.path.splitext(filepath)
        return f"{base}_{count}{ext}"
    
    def process_disclosure(self, disclosure: Disclosure) -> List[DownloadResult]:
        """Process a single disclosure and download its attachments."""
        results = []
        
        try:
            # Get attachments
            attachments, html = self.client.get_disclosure_attachments(disclosure.disclosure_id)
            
            # Filter to get latest amendment for main documents
            main_attachments = [a for a in attachments if a.is_main]
            other_attachments = [a for a in attachments if not a.is_main]
            
            if main_attachments:
                # Keep only the highest amendment level
                max_amend = max(a.amendment_level for a in main_attachments)
                main_attachments = [a for a in main_attachments if a.amendment_level == max_amend]
            
            all_attachments = main_attachments + other_attachments
            is_single = len(all_attachments) == 1
            
            if not all_attachments:
                # Zero-attachment filing - render page as PDF
                pdf_content = render_page_as_pdf(html, disclosure)
                if pdf_content:
                    folder = build_folder_path(disclosure.ticker, disclosure.disclosed_date, disclosure.form_number)
                    filename = build_filename(
                        disclosure.ticker,
                        disclosure.disclosed_date,
                        disclosure.form_number,
                        '',
                        True
                    )
                    filepath = os.path.join(folder, filename)
                    filepath = self._get_unique_filename(filepath)
                    
                    results.append(DownloadResult(
                        filepath=filepath,
                        content=pdf_content,
                        ticker=disclosure.ticker,
                        filing_type=disclosure.form_number,
                        date=disclosure.disclosed_date,
                        is_rendered=True
                    ))
                    self.log('warning', disclosure.ticker, disclosure.form_number,
                            disclosure.disclosed_date, filename)
            else:
                # Download each attachment
                for attachment in all_attachments:
                    content = self.client.download_attachment(attachment.attachment_id)
                    if content:
                        folder = build_folder_path(disclosure.ticker, disclosure.disclosed_date, disclosure.form_number)
                        filename = build_filename(
                            disclosure.ticker,
                            disclosure.disclosed_date,
                            disclosure.form_number,
                            attachment.filename,
                            is_single
                        )
                        filepath = os.path.join(folder, filename)
                        filepath = self._get_unique_filename(filepath)
                        
                        results.append(DownloadResult(
                            filepath=filepath,
                            content=content,
                            ticker=disclosure.ticker,
                            filing_type=disclosure.form_number,
                            date=disclosure.disclosed_date,
                            is_rendered=False
                        ))
                        self.log('success', disclosure.ticker, disclosure.form_number,
                                disclosure.disclosed_date, filename)
                    else:
                        self.errors.append(ErrorRecord(
                            ticker=disclosure.ticker,
                            filing_type=disclosure.form_number,
                            date=disclosure.disclosed_date,
                            error_type='download_failed',
                            message=f'Failed to download: {attachment.filename}'
                        ))
        except Exception as e:
            self.errors.append(ErrorRecord(
                ticker=disclosure.ticker,
                filing_type=disclosure.form_number,
                date=disclosure.disclosed_date,
                error_type='processing_error',
                message=str(e)
            ))
        
        return results
    
    def run(
        self,
        company_ids: List[str],
        start_date: str,
        end_date: str,
        form_numbers: List[str],
        resume: bool = False
    ) -> Tuple[List[DownloadResult], List[ErrorRecord]]:
        """Run the scraper for given parameters."""
        
        # Check for existing checkpoint
        if resume:
            self.checkpoint = load_checkpoint()
            if self.checkpoint:
                self.log_callback(f"Resuming from checkpoint (processed {len(self.checkpoint.processed_disclosures)} disclosures)")
                company_ids = self.checkpoint.company_ids
                start_date = self.checkpoint.start_date
                end_date = self.checkpoint.end_date
                form_numbers = self.checkpoint.form_numbers
        
        if not self.checkpoint:
            self.checkpoint = create_checkpoint(company_ids, start_date, end_date, form_numbers)
        
        # Process each company
        for comp_idx in range(self.checkpoint.current_company_index, len(company_ids)):
            company_id = company_ids[comp_idx]
            self.checkpoint.current_company_index = comp_idx
            
            # Fetch disclosures page by page
            page = self.checkpoint.current_page if comp_idx == self.checkpoint.current_company_index else 0
            
            while True:
                disclosures, total_pages = self.client.get_disclosures(
                    company_id, start_date, end_date, form_numbers, page
                )
                
                if not disclosures:
                    break
                
                for disclosure in disclosures:
                    # Skip if already processed
                    if disclosure.disclosure_id in self.checkpoint.processed_disclosures:
                        continue
                    
                    # Process disclosure
                    results = self.process_disclosure(disclosure)
                    self.downloaded_files.extend(results)
                    
                    # Update checkpoint
                    self.checkpoint.processed_disclosures.append(disclosure.disclosure_id)
                    self.checkpoint.downloaded_files.extend([r.filepath for r in results])
                    self.checkpoint.current_page = page
                    save_checkpoint(self.checkpoint)
                
                page += 1
                if page >= total_pages:
                    break
            
            # Reset page for next company
            self.checkpoint.current_page = 0
        
        return self.downloaded_files, self.errors

print("Scraper logic loaded!")

## Step 9: ZIP Assembly and Download

In [ ]:
def create_zip(results: List[DownloadResult]) -> bytes:
    """Create ZIP archive from download results."""
    buffer = io.BytesIO()
    
    with zipfile.ZipFile(buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
        for result in results:
            zf.writestr(result.filepath, result.content)
    
    buffer.seek(0)
    return buffer.getvalue()

def download_results(results: List[DownloadResult], errors: List[ErrorRecord]):
    """Create download for results."""
    if len(results) == 0:
        print("No files to download.")
        return
    
    if len(results) == 1:
        # Single file - download as PDF
        result = results[0]
        filename = os.path.basename(result.filepath)
        
        with open(f'/content/{filename}', 'wb') as f:
            f.write(result.content)
        files.download(f'/content/{filename}')
        print(f"Downloaded: {filename}")
    else:
        # Multiple files - create ZIP
        zip_content = create_zip(results)
        zip_size_mb = len(zip_content) / (1024 * 1024)
        
        if zip_size_mb > 100:
            print(f"Warning: ZIP file is {zip_size_mb:.1f} MB")
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        zip_filename = f'PSE_EDGE_Filings_{timestamp}.zip'
        
        with open(f'/content/{zip_filename}', 'wb') as f:
            f.write(zip_content)
        files.download(f'/content/{zip_filename}')
        print(f"Downloaded: {zip_filename} ({len(results)} files, {zip_size_mb:.1f} MB)")
    
    # Display error summary
    if errors:
        print(f"\n--- Error Summary ({len(errors)} errors) ---")
        for error in errors:
            print(f"  {error.ticker} | {error.filing_type} | {error.date} | {error.error_type}: {error.message}")

print("ZIP assembly loaded!")

## Step 10: User Interface

In [ ]:
class ScraperUI:
    """Interactive UI for the scraper."""
    
    def __init__(self):
        self.client = PSEEdgeClient()
        self.selected_companies: List[Company] = []
        self.selected_form_numbers: List[str] = []
        self.results: List[DownloadResult] = []
        self.errors: List[ErrorRecord] = []
        
        # Create widgets
        self._create_widgets()
    
    def _create_widgets(self):
        """Create all UI widgets."""
        # Company search
        self.company_search = widgets.Text(
            description='Search:',
            placeholder='Enter company name or ticker',
            layout=widgets.Layout(width='400px')
        )
        self.company_search.observe(self._on_company_search, names='value')
        
        self.company_list = widgets.SelectMultiple(
            description='Companies:',
            options=[],
            layout=widgets.Layout(width='500px', height='150px')
        )
        
        self.selected_label = widgets.HTML(
            value='<b>Selected: 0 companies</b>',
            layout=widgets.Layout(margin='10px 0')
        )
        
        self.add_btn = widgets.Button(description='Add Selected', button_style='primary')
        self.add_btn.on_click(self._on_add_companies)
        
        self.clear_btn = widgets.Button(description='Clear All', button_style='warning')
        self.clear_btn.on_click(self._on_clear_companies)
        
        # Date range
        today = datetime.now()
        year_ago = today - timedelta(days=365)
        
        self.start_date = widgets.DatePicker(
            description='From:',
            value=year_ago.date(),
            layout=widgets.Layout(width='250px')
        )
        
        self.end_date = widgets.DatePicker(
            description='To:',
            value=today.date(),
            layout=widgets.Layout(width='250px')
        )
        
        # Filing types
        common_options = [(f"{v[0]} ({k})", k) for k, v in COMMON_FILINGS.items()]
        rare_options = [(f"{v[0]} ({k})", k) for k, v in RARE_FILINGS.items()]
        
        self.common_types = widgets.SelectMultiple(
            description='Common:',
            options=common_options,
            layout=widgets.Layout(width='350px', height='200px')
        )
        
        self.show_all_toggle = widgets.Checkbox(
            value=False,
            description='Show all 83 filing types',
            layout=widgets.Layout(margin='10px 0')
        )
        self.show_all_toggle.observe(self._on_show_all_toggle, names='value')
        
        self.rare_types = widgets.SelectMultiple(
            description='Rare:',
            options=rare_options,
            layout=widgets.Layout(width='350px', height='150px', display='none')
        )
        
        # Progress and log
        self.log_output = widgets.Output(
            layout=widgets.Layout(width='100%', height='300px', overflow='auto', border='1px solid #ccc')
        )
        
        self.progress_bar = widgets.IntProgress(
            value=0,
            min=0,
            max=100,
            description='Progress:',
            layout=widgets.Layout(width='500px', display='none')
        )
        
        # Action buttons
        self.submit_btn = widgets.Button(
            description='Start Scraping',
            button_style='success',
            layout=widgets.Layout(width='150px')
        )
        self.submit_btn.on_click(self._on_submit)
        
        self.resume_btn = widgets.Button(
            description='Resume Previous',
            button_style='info',
            layout=widgets.Layout(width='150px')
        )
        self.resume_btn.on_click(self._on_resume)
        
        self.download_btn = widgets.Button(
            description='Download Files',
            button_style='primary',
            layout=widgets.Layout(width='150px', display='none')
        )
        self.download_btn.on_click(self._on_download)
    
    def _on_company_search(self, change):
        """Handle company search."""
        query = change['new']
        companies = self.client.search_companies(query)
        options = [(f"{c.ticker} - {c.company_name}", c) for c in companies]
        self.company_list.options = options
    
    def _on_add_companies(self, _):
        """Add selected companies to list."""
        for company in self.company_list.value:
            if company not in self.selected_companies:
                self.selected_companies.append(company)
        self._update_selected_label()
    
    def _on_clear_companies(self, _):
        """Clear selected companies."""
        self.selected_companies = []
        self._update_selected_label()
    
    def _update_selected_label(self):
        """Update the selected companies label."""
        tickers = ', '.join(c.ticker for c in self.selected_companies)
        count = len(self.selected_companies)
        if tickers:
            self.selected_label.value = f'<b>Selected ({count}): {tickers}</b>'
        else:
            self.selected_label.value = '<b>Selected: 0 companies</b>'
    
    def _on_show_all_toggle(self, change):
        """Toggle visibility of rare filing types."""
        if change['new']:
            self.rare_types.layout.display = 'flex'
        else:
            self.rare_types.layout.display = 'none'
    
    def _log(self, message: str):
        """Log message to output."""
        with self.log_output:
            status = 'success'
            if '[warning]' in message.lower():
                status = 'warning'
            elif 'error' in message.lower():
                status = 'danger'
            
            icon = 'check' if status == 'success' else ('exclamation' if status == 'warning' else 'times')
            print(message.replace('[success]', '').replace('[warning]', '').replace('[error]', ''))
    
    def _on_submit(self, _):
        """Start scraping."""
        self._run_scraper(resume=False)
    
    def _on_resume(self, _):
        """Resume from checkpoint."""
        self._run_scraper(resume=True)
    
    def _run_scraper(self, resume: bool):
        """Run the scraper."""
        # Validate inputs
        if not resume and not self.selected_companies:
            with self.log_output:
                print("Please select at least one company.")
            return
        
        # Get selected form numbers
        form_numbers = list(self.common_types.value) + list(self.rare_types.value)
        if not resume and not form_numbers:
            with self.log_output:
                print("Please select at least one filing type.")
            return
        
        # Clear previous log
        self.log_output.clear_output()
        
        # Show progress
        self.progress_bar.layout.display = 'flex'
        self.download_btn.layout.display = 'none'
        self.submit_btn.disabled = True
        self.resume_btn.disabled = True
        
        # Run scraper
        scraper = PSEEdgeScraper(log_callback=self._log)
        
        try:
            if resume:
                self.results, self.errors = scraper.run([], '', '', [], resume=True)
            else:
                company_ids = [c.company_id for c in self.selected_companies]
                start = self.start_date.value.strftime('%Y-%m-%d')
                end = self.end_date.value.strftime('%Y-%m-%d')
                self.results, self.errors = scraper.run(company_ids, start, end, form_numbers)
            
            # Show results
            with self.log_output:
                print(f"\n--- Complete ---")
                print(f"Downloaded: {len(self.results)} files")
                print(f"Errors: {len(self.errors)}")
            
            if self.results:
                self.download_btn.layout.display = 'flex'
            
            # Clear checkpoint on success
            clear_checkpoint()
            
        except Exception as e:
            with self.log_output:
                print(f"Error: {e}")
        finally:
            self.submit_btn.disabled = False
            self.resume_btn.disabled = False
    
    def _on_download(self, _):
        """Download results."""
        download_results(self.results, self.errors)
    
    def display(self):
        """Display the UI."""
        # Load initial company list
        companies = self.client.search_companies('')
        options = [(f"{c.ticker} - {c.company_name}", c) for c in companies]
        self.company_list.options = options
        
        # Check for existing checkpoint
        checkpoint = load_checkpoint()
        checkpoint_info = ''
        if checkpoint:
            checkpoint_info = f"""<div style="background:#e3f2fd;padding:10px;margin:10px 0;border-radius:5px;">
            Found checkpoint from {checkpoint.updated_at[:16]} with {len(checkpoint.processed_disclosures)} processed disclosures.
            Click "Resume Previous" to continue.
            </div>"""
        
        # Build layout
        display(HTML('''
        <style>
            .widget-label { font-weight: bold; }
            .section-header { font-size: 16px; font-weight: bold; margin: 15px 0 10px 0; }
        </style>
        '''))
        
        display(HTML('<h2>PSE EDGE Filing Scraper</h2>'))
        
        if checkpoint_info:
            display(HTML(checkpoint_info))
        
        # Company selection
        display(HTML('<div class="section-header">1. Select Companies</div>'))
        display(self.company_search)
        display(self.company_list)
        display(widgets.HBox([self.add_btn, self.clear_btn]))
        display(self.selected_label)
        
        # Date range
        display(HTML('<div class="section-header">2. Select Date Range</div>'))
        display(widgets.HBox([self.start_date, self.end_date]))
        
        # Filing types
        display(HTML('<div class="section-header">3. Select Filing Types</div>'))
        display(self.common_types)
        display(self.show_all_toggle)
        display(self.rare_types)
        
        # Actions
        display(HTML('<div class="section-header">4. Run Scraper</div>'))
        display(widgets.HBox([self.submit_btn, self.resume_btn, self.download_btn]))
        display(self.progress_bar)
        
        # Log output
        display(HTML('<div class="section-header">Progress Log</div>'))
        display(self.log_output)

print("UI loaded!")

---

## Run the Scraper

Execute the cell below to launch the interactive interface.

In [ ]:
# Launch the scraper UI
ui = ScraperUI()
ui.display()